# Two follow-up tokenization variants for NER and POS on Colab

Both use LoRA (r=8) on `Davlan/afro-xlmr-large-76L` with exactly the settings of the LoRA baseline (6 languages,
3 seeds, 18 runs per task). Each run is tested on all 6 languages. Only the input side changes:

* **`inner`: Morfessor cuts inside words.** The first Morfessor attempt fed every morpheme as a separate word and
  lost 7.0 (NER) and 3.4 (POS) points. Here a word stays one word; only its subword pieces are forced to stop at
  Morfessor's morpheme boundaries (`ngiya|bonga`: `▁ng iya bon ga` instead of `▁ng iya bong a`).
  Results: `results/<task>_morfessor_inner`.
* **`fvt`: Serengeti's vocabulary moved into AfroXLMR (option B).** The tokenizer is replaced by Serengeti's
  (fewest tokens per word for all six languages). Tokens AfroXLMR also has keep their vectors; new tokens get the
  mean of the AfroXLMR vectors of their pieces (Fast Vocabulary Transfer). Embeddings stay frozen, so LoRA trains
  the same parameters as the baseline. Results: `results/<task>_serengeti_fvt`.

**Before you start:** Runtime > Change runtime type > **L4 GPU**. Then Runtime > Run all.

**Time on an L4 (estimate):** about 1.5 hours per variant (NER about 1 hour, POS about 20 minutes), so about
3 hours for both. To run only one, change `VARIANTS` in step 2.

**If Colab disconnects:** reconnect and Run all again. Results are written to Google Drive after every run,
and finished runs are skipped.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 GPU, then run again."

## 2. Choose variants and mount Google Drive (results are saved here)

In [ ]:
VARIANTS = ["inner", "fvt"]   # remove one to run only the other

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/typology-aware-peft")
TASKS = ["ner", "pos"]
FVT_MODEL = "/content/afroxlmr_serengeti_fvt"
SEG = "results/morfessor_seg"
NAME = {"inner": "morfessor_inner", "fvt": "serengeti_fvt"}
ARGS = {"inner": f"--segment {SEG} --seg_mode inner", "fvt": f"--model {FVT_MODEL}"}
OUT = {(v, t): DRIVE / "results" / f"{t}_{NAME[v]}" for v in VARIANTS for t in TASKS}
for p in OUT.values():
    p.mkdir(parents=True, exist_ok=True)
print("Results folders:", *OUT.values(), sep="\n  ")

## 3. Get the code (latest version from GitHub)

In [ ]:
import os, shutil
REPO_URL = "https://github.com/WanjaWhoopie/typology-aware-peft.git"
WORK = Path("/content/typology-aware-peft")
if WORK.exists():
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO_URL} {WORK}
for name in ["src/token_baseline.py", "src/token_report.py", "src/sib200_report.py", "src/vocab_transfer.py",
             *(f"results/morfessor_seg/{l}.json" for l in ["amh", "hau", "kin", "lug", "swa", "zul"])]:
    assert (WORK / name).exists(), f"{name} is not on GitHub yet; push it first"
os.chdir(WORK)
!git log --oneline -1

## 4. Install packages

Same pinned versions as the SIB-200 run. Colab now ships an old `torchao` that makes peft crash when it
builds a LoRA model (this stopped every LoRA run on 2 Nov), so it is removed; nothing here uses it.

In [ ]:
!pip install -q transformers==4.57.2 datasets==4.4.1 peft==0.21.2 scikit-learn
!pip uninstall -y -q torchao
import transformers, datasets, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

## 5. Build the Serengeti-vocabulary model (fvt only, a few minutes)

Saved to `/content`, not to Drive (2.2 GB). After a disconnect this cell simply builds it again; the result is
identical every time, so finished runs stay valid.

In [ ]:
if "fvt" in VARIANTS and not Path(FVT_MODEL, "model.safetensors").exists():
    !python src/vocab_transfer.py --out {FVT_MODEL}
    assert _exit_code == 0, "vocabulary transfer failed; see the error above"
if "fvt" in VARIANTS:
    print(Path(FVT_MODEL, "transfer_stats.json").read_text())

## 6. Smoke test (a few minutes)

2 languages, 64 sentences, 1 epoch, per variant and task. Only checks that everything runs.
Scores near zero and "collapsed" warnings are normal here. The `inner` smoke test uses the small base model;
the `fvt` one uses the real transferred model.

In [ ]:
for v in VARIANTS:
    for t in TASKS:
        a = ARGS[v] if v == "fvt" else ARGS[v] + " --model Davlan/afro-xlmr-base"
        smoke_out = f"/content/smoke_{v}_{t}"
        !python src/token_baseline.py --task {t} --smoke {a} --out {smoke_out}
        assert _exit_code == 0, f"smoke test {v} {t} failed; see the error above"

## 7. Full runs: NER then POS for each variant (LoRA)

* **Out of memory:** rerun this cell with `--bs 8` added (and write down that the batch size changed).
* **Collapsed run** (dev score below the threshold): let it finish, then use step 9.

In [ ]:
for v in VARIANTS:
    for t in TASKS:
        a, out = ARGS[v], OUT[(v, t)]
        print(f"===== {v} {t} =====")
        !python src/token_baseline.py --task {t} --methods lora {a} --out {out}

## 8. Check the runs

In [ ]:
import json
def show(v, task):
    f = OUT[(v, task)] / "runs.jsonl"
    runs = [json.loads(l) for l in f.read_text().splitlines() if l.strip()] if f.exists() else []
    print(f"{v} {task.upper()}: {len(runs)} of 18 runs finished")
    print(f"{'seed':>4} {'src':4} {'dev':>6} {'epoch':>5} {'min':>5} {'GB':>5}  collapsed")
    for r in sorted(runs, key=lambda r: (r["src"], r["seed"])):
        print(f"{r['seed']:>4} {r['src']:4} {r['best_dev']:>6.3f} {r['best_epoch']:>5} "
              f"{r['seconds'] / 60:>5.1f} {r['peak_mem_gb'] or 0:>5.1f}  {'YES' if r['collapsed'] else ''}")
    print()
    return runs
runs = {(v, t): show(v, t) for v in VARIANTS for t in TASKS}

## 9. Optional: redo collapsed runs with a lower learning rate

Only if step 8 shows collapsed runs. Set `REDO = True`, run this cell, then run step 8 again.
The new learning rate is stored in `runs.jsonl`, so mention it in the write-up.

In [ ]:
REDO = False

def key(r):
    return (r["method"], r["seed"], r["src"])

if REDO:
    for (v, t), rs in runs.items():
        bad = {key(r) for r in rs if r["collapsed"]}
        if not bad:
            continue
        print(v, t, "redoing:", sorted(bad))
        for fname in ("runs.jsonl", "results.jsonl"):
            f = OUT[(v, t)] / fname
            keep = [l for l in f.read_text().splitlines() if l.strip() and key(json.loads(l)) not in bad]
            f.write_text("\n".join(keep) + "\n")
        a, out = ARGS[v], OUT[(v, t)]
        !python src/token_baseline.py --task {t} --methods lora {a} --out {out} --lr_lora 1.5e-4

## 10. Reports

In [ ]:
from IPython.display import Markdown, display
for (v, t), out in OUT.items():
    !python src/token_report.py --task {t} --dir {out} > /dev/null
    display(Markdown(f"# {v}\n\n" + (out / "summary.md").read_text()))

## 11. Download the results

One zip per variant and task, e.g. `ner_morfessor_inner_results.zip`, `pos_serengeti_fvt_results.zip`.
Leave them in Downloads and tell Claude. The same files also stay on Google Drive in `typology-aware-peft/results/`.

In [ ]:
from google.colab import files
for (v, t), out in OUT.items():
    z = f"/content/{t}_{NAME[v]}_results"
    shutil.make_archive(z, "zip", out)
    files.download(z + ".zip")